In [2]:
import torch
from torch.utils.data import Dataset
from PIL import Image
import pandas as pd
import os

In [3]:
from transformers import TrOCRProcessor, VisionEncoderDecoderModel

d:\Files\AI 3.5c rep\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
from transformers import AutoTokenizer
import re
from collections import Counter

In [5]:
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments
from datasets import Dataset as HFDataset
import numpy as np

In [6]:
class Im2LatexDataset(Dataset):
    def __init__(self, image_dir, formulas_csv, split_csv, processor, max_length=512):
        self.image_dir = image_dir
        self.processor = processor
        self.max_length = max_length
        
        # Загружаем словарь формул (колонка: formulas)
        formulas_df = pd.read_csv(formulas_csv)
        # Убедимся, что колонка называется 'formulas' или переименуем
        if 'formulas' not in formulas_df.columns:
            # Возможно, там другая структура, выведем для диагностики
            print(f"Колонки в formulas_csv: {formulas_df.columns.tolist()}")
            raise ValueError("Ожидалась колонка 'formulas'")
        
        # Создаём словарь: индекс строки -> формула
        self.formula_dict = {}
        for idx, row in formulas_df.iterrows():
            self.formula_dict[idx] = row['formulas']
        
        print(f"Загружено {len(self.formula_dict)} формул из {formulas_csv}")
        
        # Загружаем сплит (колонки: formula, image)
        split_df = pd.read_csv(split_csv)
        print(f"Колонки в split_csv: {split_df.columns.tolist()}")
        
        # Проверяем наличие нужных колонок
        if 'image' not in split_df.columns:
            raise ValueError(f"Ожидалась колонка 'image' в {split_csv}")
        
        # Сохраняем список имён изображений
        self.image_names = split_df['image'].tolist()
        # И соответствующие формулы (для валидации можно использовать)
        if 'formula' in split_df.columns:
            self.split_formulas = split_df['formula'].tolist()
        else:
            self.split_formulas = None
        
        print(f"Загружено {len(self.image_names)} примеров из {split_csv}")
        print(f"Пример имени изображения: {self.image_names[0]}")
    
    def __len__(self):
        return len(self.image_names)
    
    def __getitem__(self, idx):
        # Получаем имя изображения
        image_name = self.image_names[idx]
        image_path = os.path.join(self.image_dir, image_name)
        
        # Проверяем существование файла
        if not os.path.exists(image_path):
            # Может быть имя без расширения
            if not image_name.endswith('.png'):
                image_path_png = image_path + '.png'
                if os.path.exists(image_path_png):
                    image_path = image_path_png
                else:
                    raise FileNotFoundError(
                        f"Не найдено изображение: {image_path} или {image_path_png}"
                    )
        
        # Загружаем изображение
        try:
            image = Image.open(image_path).convert("RGB")
        except Exception as e:
            raise RuntimeError(f"Ошибка загрузки изображения {image_path}: {e}")
        
        # Получаем формулу из split_csv (если есть) или по имени файла
        if self.split_formulas is not None:
            formula = self.split_formulas[idx]
        else:
            # Ищем в словаре по индексу (из имени файла)
            # Извлекаем число из имени файла (например, "45.png" -> 45)
            base_name = os.path.splitext(image_name)[0]
            try:
                formula_idx = int(base_name)
            except ValueError:
                # Если имя не число, ищем в словаре по ключу
                raise ValueError(
                    f"Не удалось определить индекс формулы из имени файла '{image_name}'. "
                    f"Имя файла должно быть числом или должна быть колонка 'formula' в split_csv."
                )
            
            if formula_idx not in self.formula_dict:
                raise KeyError(f"Формула с индексом {formula_idx} не найдена в словаре")
            formula = self.formula_dict[formula_idx]
        
        # Обрабатываем через processor
        encoding = self.processor(
            images=image,
            text=formula,
            return_tensors="pt",
            padding="max_length",
            truncation=True,
            max_length=self.max_length
        )
        
        # Убираем лишнюю размерность батча
        encoding = {k: v.squeeze(0) for k, v in encoding.items()}
        
        # Выводим ключи для отладки (только первый раз)
        if idx == 0:
            print(f"DEBUG: Ключи в encoding: {encoding.keys()}")
            for key in encoding.keys():
                print(f"DEBUG: {key} shape: {encoding[key].shape}")
        
        # TrOCR processor возвращает labels вместо input_ids
        # Создаём input_ids из labels если нужно
        if 'labels' in encoding and 'input_ids' not in encoding:
            encoding['input_ids'] = encoding['labels'].clone()
        # Или наоборот
        elif 'input_ids' in encoding and 'labels' not in encoding:
            encoding['labels'] = encoding['input_ids'].clone()
        elif 'input_ids' in encoding and 'labels' in encoding:
            # Уже есть оба, ничего не делаем
            pass
        else:
            # Если нет ни того ни другого, пробуем другие ключи
            available_keys = list(encoding.keys())
            raise KeyError(
                f"Не найдены 'input_ids' или 'labels' в encoding. "
                f"Доступные ключи: {available_keys}"
            )
        
        return encoding


# ===== Функция для проверки датасета =====
def test_dataset():
    """Тестируем загрузчик перед обучением"""
    from transformers import TrOCRProcessor
    
    print("=" * 50)
    print("ТЕСТИРОВАНИЕ ЗАГРУЗЧИКА ДАННЫХ")
    print("=" * 50)
    
    # Загружаем процессор
    processor = TrOCRProcessor.from_pretrained("microsoft/trocr-small-printed")
    print("✓ Процессор загружен")
    
    # Создаём датасет
    dataset = Im2LatexDataset(
        image_dir="im2latex-100k\\formula_images_processed",
        formulas_csv="im2latex-100k\\im2latex_formulas.norm.csv",
        split_csv="im2latex-100k\\im2latex_train.csv",
        processor=processor,
        max_length=256
    )
    print("✓ Датасет создан")
    
    # Проверяем первый элемент
    print("\nПроверка первого элемента:")
    sample = dataset[0]
    print(f"  pixel_values shape: {sample['pixel_values'].shape}")
    print(f"  input_ids shape: {sample['input_ids'].shape}")
    print(f"  Декодированная формула: {processor.decode(sample['input_ids'], skip_special_tokens=True)}")
    
    # Проверяем несколько случайных элементов
    print("\nПроверка случайных элементов:")
    import random
    for i in random.sample(range(len(dataset)), min(3, len(dataset))):
        sample = dataset[i]
        formula = processor.decode(sample['input_ids'], skip_special_tokens=True)
        print(f"  [{i}] {formula[:80]}...")
    
    print("\n✓ Все проверки пройдены!")
    return dataset

In [7]:
dataset = test_dataset()

ТЕСТИРОВАНИЕ ЗАГРУЗЧИКА ДАННЫХ


✓ Процессор загружен
Загружено 102863 формул из im2latex-100k\im2latex_formulas.norm.csv
Колонки в split_csv: ['formula', 'image']
Загружено 75275 примеров из im2latex-100k\im2latex_train.csv
Пример имени изображения: 66667cee5b.png
✓ Датасет создан

Проверка первого элемента:
DEBUG: Ключи в encoding: dict_keys(['pixel_values', 'labels'])
DEBUG: pixel_values shape: torch.Size([3, 384, 384])
DEBUG: labels shape: torch.Size([256])
  pixel_values shape: torch.Size([3, 384, 384])
  input_ids shape: torch.Size([256])
  Декодированная формула: \widetilde \gamma _ { \mathrm { h o p f } } \simeq \sum _ { n > 0 } \widetilde { G } _ { n } { \frac { ( - a ) ^ { n } } { 2 ^ { 2 n - 1 } } }

Проверка случайных элементов:
  [25510] \{ C , D \} _ { ( 3 , 2 ) } = 4 \beta { \gamma } ^ { 2 } ( c { \partial } _ { 2 ...
  [27116] \left( V , 1 , 1 \right) = \left( \begin{array} { c c c } { 1 + \frac { V ^ { 2 ...
  [17364] \Delta _ { g h } = - \nabla ^ { 2 } + \xi q ^ { 2 } \phi _ { c l } ^ { 2 }...

✓ Все

In [8]:
print("=" * 50)
print("ДИАГНОСТИКА CSV-ФАЙЛОВ")
print("=" * 50)

# 1. Проверяем formulas.norm.csv
print("\n1. im2latex_formulas.norm.csv")
formulas_df = pd.read_csv("im2latex-100k\\im2latex_formulas.norm.csv")
print(f"   Колонки: {formulas_df.columns.tolist()}")
print(f"   Размер: {formulas_df.shape}")
print(f"   Первые 3 строки:")
print(formulas_df.head(3).to_string())

# 2. Проверяем train.csv
print("\n2. im2latex_train.csv")
train_df = pd.read_csv("im2latex-100k\\im2latex_train.csv")
print(f"   Колонки: {train_df.columns.tolist()}")
print(f"   Размер: {train_df.shape}")
print(f"   Первые 3 строки:")
print(train_df.head(3).to_string())

# 3. Проверяем наличие изображений
print("\n3. Проверка изображений")
image_dir = "im2latex-100k\\formula_images_processed"
if os.path.exists(image_dir):
    images = os.listdir(image_dir)
    print(f"   Всего изображений: {len(images)}")
    print(f"   Первые 5: {images[:5]}")
    
    # Проверяем, что изображения из train.csv существуют
    print(f"\n   Проверка первых 5 изображений из train.csv:")
    for img_name in train_df['image'].head(5):
        img_path = os.path.join(image_dir, img_name)
        exists = os.path.exists(img_path)
        print(f"   {'✓' if exists else '✗'} {img_name}")
else:
    print(f"   ✗ Папка {image_dir} не найдена!")
    print(f"   Текущая директория: {os.getcwd()}")
    print(f"   Содержимое: {os.listdir('.')}")

# 4. Проверяем соответствие формул
print("\n4. Соответствие формул")
if 'formula' in train_df.columns:
    print("   В train.csv есть колонка 'formula' — формулы берутся оттуда")
    print(f"   Пример: {train_df['formula'].iloc[0]}")
else:
    print("   Колонки 'formula' нет в train.csv — формулы будут искаться по имени файла в formulas.norm.csv")

ДИАГНОСТИКА CSV-ФАЙЛОВ

1. im2latex_formulas.norm.csv
   Колонки: ['formulas']
   Размер: (102863, 1)
   Первые 3 строки:
                                                                                                                                                                                                                                                                                                                                      formulas
0  \int _ { - \epsilon } ^ { \infty } d l \: \mathrm { e } ^ { - l \zeta } \int _ { - \epsilon } ^ { \infty } d l ^ { \prime } \mathrm { e } ^ { - l ^ { \prime } \zeta } l l ^ { \prime } { \frac { l ^ { \prime } - l } { l + l ^ { \prime } } } \{ 3 \, \delta ^ { \prime \prime } ( l ) - { \frac { 3 } { 4 } } t \, \delta ( l ) \} = 0 .
1       d s ^ { 2 } = ( 1 - { \frac { q c o s \theta } { r } } ) ^ { \frac { 2 } { 1 + \alpha ^ { 2 } } } \lbrace d r ^ { 2 } + r ^ { 2 } d \theta ^ { 2 } + r ^ { 2 } s i n ^ { 2 } \theta d \varphi ^ { 2 } \r

In [11]:
def quick_train_test():
    """Минимальное обучение для проверки пайплайна"""
    from transformers import (
        TrOCRProcessor,
        VisionEncoderDecoderModel,
        Seq2SeqTrainer,
        Seq2SeqTrainingArguments
    )
    
    print("=" * 50)
    print("ТЕСТОВОЕ ОБУЧЕНИЕ (100 примеров)")
    print("=" * 50)
    
    # Загружаем модель
    model_name = "microsoft/trocr-small-printed"
    processor = TrOCRProcessor.from_pretrained(model_name)
    model = VisionEncoderDecoderModel.from_pretrained(model_name)
    print("✓ Модель загружена")
    
    # Настройки модели
    model.config.decoder_start_token_id = processor.tokenizer.cls_token_id
    model.config.pad_token_id = processor.tokenizer.pad_token_id
    model.generation_config.max_length = 256
    
    # Создаём датасет
    full_dataset = Im2LatexDataset(
        image_dir="im2latex-100k\\formula_images_processed",
        formulas_csv="im2latex-100k\\im2latex_formulas.norm.csv",
        split_csv="im2latex-100k\\im2latex_train.csv",
        processor=processor,
        max_length=256
    )
    
    # Берём 100 примеров для обучения и 20 для валидации
    train_dataset = torch.utils.data.Subset(full_dataset, range(100))
    
    val_dataset = Im2LatexDataset(
        image_dir="im2latex-100k\\formula_images_processed",
        formulas_csv="im2latex-100k\\im2latex_formulas.norm.csv",
        split_csv="im2latex-100k\\im2latex_validate.csv",
        processor=processor,
        max_length=256
    )
    val_dataset = torch.utils.data.Subset(val_dataset, range(20))
    
    print(f"✓ Обучающая выборка: {len(train_dataset)} примеров")
    print(f"✓ Валидационная выборка: {len(val_dataset)} примеров")
    
    # Аргументы обучения
    training_args = Seq2SeqTrainingArguments(
        output_dir="./test_training_output",
        per_device_train_batch_size=2,
        per_device_eval_batch_size=2,
        num_train_epochs=3,
        logging_steps=10,
        save_steps=50,
        eval_strategy="steps",
        eval_steps=50,
        save_total_limit=2,
        fp16=False,
        predict_with_generate=True,
        generation_max_length=256,
        report_to="none",  # отключаем wandb
    )
    
    # Тренер
    trainer = Seq2SeqTrainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
    )
    
    # Запускаем обучение
    print("\nНачинаем тестовое обучение...")
    trainer.train()
    print("\n✓ Тестовое обучение завершено!")
    
    # Сохраняем модель
    model.save_pretrained("./test_trained_model")
    processor.save_pretrained("./test_trained_model")
    print("✓ Модель сохранена в ./test_trained_model")
    
    return trainer, model, processor

In [12]:
trainer, model, processor = quick_train_test()

ТЕСТОВОЕ ОБУЧЕНИЕ (100 примеров)


Loading weights: 100%|██████████| 360/360 [00:00<00:00, 10457.15it/s]
[transformers] VisionEncoderDecoderModel LOAD REPORT from: microsoft/trocr-small-printed
Key                         | Status  | 
----------------------------+---------+-
encoder.pooler.dense.bias   | MISSING | 
encoder.pooler.dense.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


✓ Модель загружена
Загружено 102863 формул из im2latex-100k\im2latex_formulas.norm.csv
Колонки в split_csv: ['formula', 'image']
Загружено 75275 примеров из im2latex-100k\im2latex_train.csv
Пример имени изображения: 66667cee5b.png
Загружено 102863 формул из im2latex-100k\im2latex_formulas.norm.csv
Колонки в split_csv: ['formula', 'image']
Загружено 8370 примеров из im2latex-100k\im2latex_validate.csv
Пример имени изображения: 5abbb9b19f.png
✓ Обучающая выборка: 100 примеров
✓ Валидационная выборка: 20 примеров

Начинаем тестовое обучение...


d:\Files\AI 3.5c rep\venv\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss,Validation Loss
50,1.369492,0.982745
100,1.135208,0.858047
150,1.012196,0.820609


...

DEBUG: Ключи в encoding: dict_keys(['pixel_values', 'labels'])
DEBUG: pixel_values shape: torch.Size([3, 384, 384])
DEBUG: labels shape: torch.Size([256])
DEBUG: Ключи в encoding: dict_keys(['pixel_values', 'labels'])
DEBUG: pixel_values shape: torch.Size([3, 384, 384])
DEBUG: labels shape: torch.Size([256])


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  4.15it/s]



✓ Тестовое обучение завершено!


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.14it/s]

✓ Модель сохранена в ./test_trained_model


In [13]:
def predict_formula(image_path, model, processor):
    """Распознавание одной формулы"""
    image = Image.open(image_path).convert("RGB")
    
    pixel_values = processor(images=image, return_tensors="pt").pixel_values
    
    # Перемещаем на то же устройство, что и модель
    device = next(model.parameters()).device
    pixel_values = pixel_values.to(device)
    
    generated_ids = model.generate(
        pixel_values,
        max_length=256,
        num_beams=4,
        early_stopping=True
    )
    
    generated_text = processor.batch_decode(
        generated_ids,
        skip_special_tokens=True
    )[0]
    
    return generated_text


In [14]:
formula = predict_formula("test_formulas.jpg", model, processor)
print(formula)